In [ ]:
import pandas as pd
import yaml
import pycountry

### File paths

In [ ]:
production_fn = '../../resources/steel_production.csv'
bus_locations_fn = '../../data/bus_locations.csv'

# Outputs
production_clustered_fn = '../../resources/steel_production_clustered.csv'

### Get steel data

In [ ]:

# Read the CSV data
production = pd.read_csv(production_fn, index_col='ISO_A2')

In [ ]:
production

### Cluster per region

### Cluster countries per region
Map each country in the production data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config file
with open('../../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
regions = config['regions']

# Build a mapping from country name to ISO_A2 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_2
    # Add common names
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_2

# Build a mapping from ISO_A2 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        # Some country names may have extra text (e.g., "Togo + Algeria"), handle them simply
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

# Map each row in production to its region
production['region'] = production.index.map(lambda iso: iso_to_region.get(iso, 'Other'))
production.sort_values(by='SteelProductionMt', ascending=False)

### Aggregate steel production by region

In [ ]:
# Group by region and sum all numeric columns
production_by_region = production.groupby('region').sum(numeric_only=True)
production_by_region

### Save clustered steel production

In [ ]:
production_by_region.to_csv(production_clustered_fn)